In [1]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import json
import hashlib
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import pandas as pd
import torch
from huggingface_hub import HfApi

BASELINE_ROOT = Path(
    "/kaggle/input/datasets/muusamunufo/base-1-trained/"
    "HealthVer_Experiments"
)
OUTPUT_DIR = Path("/kaggle/working/HealthVer_SmolLM2_Experiments")
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert BASELINE_ROOT.is_dir(), "Attach the base-1-trained dataset."

dataset_metadata = json.loads(
    (BASELINE_ROOT / "dataset_metadata.json").read_text()
)
baseline_input_config = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
baseline_run_config = json.loads(
    (BASELINE_ROOT / "Qwen_LoRA_full_seed42" / "run_config.json").read_text()
)

DATASET_REVISION = dataset_metadata["revision"]
assert DATASET_REVISION == "b20ac99ceed62f5264a319fa25a854df1668d85b"

raw_frames = {}
for split, filename, expected_rows in [
    ("train", "healthver_train.csv", 10590),
    ("validation", "healthver_dev.csv", 1917),
]:
    path = BASELINE_ROOT / "original_data" / (
        f"{DATASET_REVISION}_{filename}"
    )
    assert sha256(path) == dataset_metadata["files"][split]["sha256"]
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(frame) == expected_rows
    assert frame["id"].is_unique
    assert set(frame["label"]) == {"Supports", "Refutes", "Neutral"}
    raw_frames[split] = frame
    print(f"{split}: {len(frame):,} rows; original CSV hash verified")

test_path = BASELINE_ROOT / "original_data" / (
    f"{DATASET_REVISION}_healthver_test.csv"
)
assert test_path.is_file()  # Presence only; test contents not read.

print("\nPython:", sys.version.split()[0])
for package in (
    "torch", "transformers", "peft", "accelerate",
    "datasets", "scikit-learn", "huggingface-hub", "torchao",
):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {properties.name}, "
        f"{properties.total_memory / 1024**3:.2f} GiB"
    )

model_info = HfApi().model_info(MODEL_ID)
MODEL_REVISION = model_info.sha
assert isinstance(MODEL_REVISION, str) and len(MODEL_REVISION) == 40

print("\nNext model:", MODEL_ID)
print("Resolved model revision:", MODEL_REVISION)
print("Planned output:", OUTPUT_DIR)
print("\nCheck complete.")
print("No weights loaded, packages changed, or output files written.")
print("Test contents not accessed.")

train: 10,590 rows; original CSV hash verified
validation: 1,917 rows; original CSV hash verified

Python: 3.13.15
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
datasets: 4.8.5
scikit-learn: 1.6.1
huggingface-hub: 1.29.0
torchao: NOT INSTALLED
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB

Next model: HuggingFaceTB/SmolLM2-360M-Instruct
Resolved model revision: a10cc1512eabd3dde888204e902eca88bddb4951
Planned output: /kaggle/working/HealthVer_SmolLM2_Experiments

Check complete.
No weights loaded, packages changed, or output files written.
Test contents not accessed.


In [3]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# Pin the revision for reproducible future Run All executions.
MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
LABELS = ["Supports", "Refutes", "Neutral"]
SYSTEM_PROMPT = baseline_input_config["system_prompt"]
CHECK_LIMIT = 384

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)
assert tokenizer.chat_template, "Missing native chat template."
assert tokenizer.eos_token_id is not None

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def make_messages(claim, evidence):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

def chat_ids(messages, generation_prompt=False):
    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=generation_prompt,
        return_dict=False,
    )
    assert isinstance(ids, list)
    assert all(isinstance(x, int) for x in ids)
    return ids

sample = raw_frames["train"].iloc[0]
messages = make_messages(sample["claim"], sample["evidence"])
prompt_ids = chat_ids(messages, generation_prompt=True)

print("Assistant label encodings:")
for label in LABELS:
    completed = chat_ids(
        messages + [{"role": "assistant", "content": label}]
    )
    assert completed[:len(prompt_ids)] == prompt_ids
    answer = completed[len(prompt_ids):]
    assert tokenizer.eos_token_id in answer
    print(label, answer, repr(tokenizer.decode(
        answer, skip_special_tokens=False
    )))

length_reports = []
for split, frame in raw_frames.items():
    lengths, prompt_lengths = [], []

    for row in tqdm(
        frame.itertuples(index=False),
        total=len(frame),
        desc=f"Checking {split}",
    ):
        messages = make_messages(row.claim, row.evidence)
        prompt = chat_ids(messages, generation_prompt=True)
        completed = chat_ids(
            messages + [{"role": "assistant", "content": row.label}]
        )

        assert completed[:len(prompt)] == prompt, (
            f"Prompt boundary mismatch: {split}, ID={row.id}"
        )
        answer = completed[len(prompt):]
        assert answer and tokenizer.eos_token_id in answer

        lengths.append(len(completed))
        prompt_lengths.append(len(prompt))

    length_reports.append({
        "split": split,
        "rows": len(frame),
        "max_complete_tokens": max(lengths),
        "complete_rows_over_384": sum(n > CHECK_LIMIT for n in lengths),
        "max_prompt_plus_12": max(prompt_lengths) + 12,
    })

length_report = pd.DataFrame(length_reports)
print("\n", length_report.to_string(index=False))
print("\nAll prompt boundaries and answer EOS checks passed.")
print("No truncation or row removal performed.")
print("No model weights loaded or experiment files written.")
print("Test contents not accessed.")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Assistant label encodings:
Supports [12233, 1875, 2, 198] 'Supports<|im_end|>\n'
Refutes [7140, 2562, 2, 198] 'Refutes<|im_end|>\n'
Neutral [9042, 22728, 2, 198] 'Neutral<|im_end|>\n'


Checking train:   0%|          | 0/10590 [00:00<?, ?it/s]

Checking validation:   0%|          | 0/1917 [00:00<?, ?it/s]


      split  rows  max_complete_tokens  complete_rows_over_384  max_prompt_plus_12
     train 10590                  339                       0                 347
validation  1917                  328                       0                 336

All prompt boundaries and answer EOS checks passed.
No truncation or row removal performed.
No model weights loaded or experiment files written.
Test contents not accessed.


In [4]:
import json
import hashlib
from datasets import Dataset
from tqdm.auto import tqdm

SEQUENCE_LIMIT = 384
assert MODEL_REVISION == "a10cc1512eabd3dde888204e902eca88bddb4951"

input_config = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "system_prompt": SYSTEM_PROMPT,
    "labels": LABELS,
    "truncation": False,
    "max_sequence_length": SEQUENCE_LIMIT,
    "padding": "dynamic",
    "loss": "assistant_answer_only",
    "chat_template_sha256": hashlib.sha256(
        tokenizer.chat_template.encode("utf-8")
    ).hexdigest(),
}

# Protect existing artifacts from accidental replacement.
artifact_names = [
    "tokenized_train", "tokenized_validation", "tokenizer",
    "input_config.json", "preprocessing_manifest.json",
]
for name in artifact_names:
    assert not (OUTPUT_DIR / name).exists(), (
        f"Already exists; stop and inspect: {OUTPUT_DIR / name}"
    )

prepared = {}
manifest = {
    "baseline_source": str(BASELINE_ROOT),
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "row_order": "original_CSV_order",
    "splits": {},
}

for split in ("train", "validation"):
    frame = raw_frames[split]
    records = []

    for row in tqdm(
        frame.itertuples(index=False),
        total=len(frame),
        desc=f"Preparing {split}",
    ):
        messages = make_messages(row.claim, row.evidence)
        prompt = chat_ids(messages, generation_prompt=True)
        ids = chat_ids(
            messages + [{"role": "assistant", "content": row.label}]
        )

        assert ids[:len(prompt)] == prompt
        answer = ids[len(prompt):]
        assert len(ids) <= SEQUENCE_LIMIT
        assert tokenizer.eos_token_id in answer
        assert tokenizer.decode(
            answer, skip_special_tokens=True
        ).strip() == row.label

        records.append({
            "id": str(row.id),
            "input_ids": ids,
            "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prompt) + answer,
            "sequence_length": len(ids),
        })

    dataset = Dataset.from_list(records)
    assert list(dataset["id"]) == frame["id"].tolist()
    prepared[split] = dataset

    manifest["splits"][split] = {
        "rows": len(dataset),
        "source_csv_sha256": dataset_metadata["files"][split]["sha256"],
        "dataset_fingerprint": dataset._fingerprint,
        "max_sequence_length": max(dataset["sequence_length"]),
        "supervised_token_counts": sorted({
            sum(token != -100 for token in r["labels"])
            for r in records
        }),
    }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split, dataset in prepared.items():
    dataset.save_to_disk(str(OUTPUT_DIR / f"tokenized_{split}"))

tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))
(OUTPUT_DIR / "input_config.json").write_text(
    json.dumps(input_config, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "preprocessing_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

full_train = prepared["train"]
full_validation = prepared["validation"]

print(json.dumps(manifest["splits"], indent=2))
print("Saved:", OUTPUT_DIR)
print("No model loaded or training performed. Test data not accessed.")

Preparing train:   0%|          | 0/10590 [00:00<?, ?it/s]

Preparing validation:   0%|          | 0/1917 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/10590 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1917 [00:00<?, ? examples/s]

{
  "train": {
    "rows": 10590,
    "source_csv_sha256": "57668949f79d0f46e66d74df4b0bc9812cad4d5ad7c5dd03dfba008a608b3f2a",
    "dataset_fingerprint": "d4c85703601925f6",
    "max_sequence_length": 339,
    "supervised_token_counts": [
      4
    ]
  },
  "validation": {
    "rows": 1917,
    "source_csv_sha256": "6f8ea31cbcf2ea5a2d3c82d65587d66e96de4f70e72642bbf7f61fec6ef8a93a",
    "dataset_fingerprint": "46957bacf30743e1",
    "max_sequence_length": 328,
    "supervised_token_counts": [
      4
    ]
  }
}
Saved: /kaggle/working/HealthVer_SmolLM2_Experiments
No model loaded or training performed. Test data not accessed.


In [5]:
import json
import torch
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model

TRAIN_SEED = 42
DEVICE = torch.device("cuda:0")
INITIAL_ADAPTER_DIR = OUTPUT_DIR / "initial_adapter_seed42"
INITIALIZATION_REPORT = OUTPUT_DIR / "initialization_manifest.json"

assert not INITIAL_ADAPTER_DIR.exists(), "Initial adapter already exists."
assert not INITIALIZATION_REPORT.exists(), "Initialization report already exists."
assert torch.cuda.is_available()

torch.cuda.set_device(DEVICE)
set_seed(TRAIN_SEED)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": "cuda:0"},
)

assert model.config.max_position_embeddings >= SEQUENCE_LIMIT
assert len(tokenizer) <= model.get_input_embeddings().num_embeddings

# Reset immediately before creating the randomly initialized adapter.
set_seed(TRAIN_SEED)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    revision=MODEL_REVISION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    init_lora_weights=True,
)

model = get_peft_model(model, lora_config)

# Match the previous precision policy:
# frozen base in FP16; trainable adapter parameters in FP32.
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        assert "lora_" in name, f"Unexpected trainable parameter: {name}"
        parameter.data = parameter.data.float()
        assert torch.isfinite(parameter).all(), name
    else:
        assert parameter.dtype == torch.float16, (name, parameter.dtype)

trainable = [(n, p) for n, p in model.named_parameters()
             if p.requires_grad]
assert trainable
assert all(p.device == DEVICE for _, p in model.named_parameters())

# Standard LoRA starts with zero B matrices, hence zero adapter update.
b_parameters = [(n, p) for n, p in trainable if "lora_B" in n]
assert b_parameters
assert all(torch.count_nonzero(p).item() == 0 for _, p in b_parameters)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.print_trainable_parameters()

model.save_pretrained(
    str(INITIAL_ADAPTER_DIR),
    safe_serialization=True,
    save_embedding_layers=False,
)

weights_path = INITIAL_ADAPTER_DIR / "adapter_model.safetensors"
assert weights_path.is_file()

report = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "initialization_seed": TRAIN_SEED,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": "all-linear",
    "base_dtype": "float16",
    "adapter_dtype": "float32",
    "trainable_parameters": sum(p.numel() for _, p in trainable),
    "total_parameters": sum(p.numel() for p in model.parameters()),
    "initial_adapter_sha256": sha256(weights_path),
    "optimizer_updates": 0,
}
INITIALIZATION_REPORT.write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

print(json.dumps(report, indent=2))
print("Saved:", INITIAL_ADAPTER_DIR)
print("No training or validation/test evaluation performed.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437
{
  "model": "HuggingFaceTB/SmolLM2-360M-Instruct",
  "model_revision": "a10cc1512eabd3dde888204e902eca88bddb4951",
  "initialization_seed": 42,
  "lora_rank": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "target_modules": "all-linear",
  "base_dtype": "float16",
  "adapter_dtype": "float32",
  "trainable_parameters": 8683520,
  "total_parameters": 370504640,
  "initial_adapter_sha256": "9019d42290483419b28d9b567277d0d8082f6a53245c4aaffc05cb331e6b68a0",
  "optimizer_updates": 0
}
Saved: /kaggle/working/HealthVer_SmolLM2_Experiments/initial_adapter_seed42
No training or validation/test evaluation performed.


In [6]:
import json
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import set_seed

def training_collator(rows):
    padding = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    batch = {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding.items()
    }
    assert batch["input_ids"].shape[1] <= SEQUENCE_LIMIT
    return batch

# Inspect the longest examples without changing dataset order.
lengths = list(full_train["sequence_length"])
check_indices = sorted(
    range(len(lengths)), key=lambda i: (-lengths[i], i)
)[:8]

batch = {
    key: value.to(DEVICE)
    for key, value in training_collator(
        [full_train[i] for i in check_indices]
    ).items()
}
assert (batch["labels"] != -100).sum(dim=1).tolist() == [4] * 8
assert (batch["labels"][batch["attention_mask"] == 0] == -100).all()

previous_mode = model.training
model.train()
model.zero_grad(set_to_none=True)
set_seed(TRAIN_SEED)
torch.cuda.synchronize(DEVICE)
torch.cuda.reset_peak_memory_stats(DEVICE)

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss = model(**batch, use_cache=False).loss

    assert torch.isfinite(loss).item(), "Non-finite loss."

    # Scale backward to reduce FP16 underflow; no optimizer is created.
    loss_scale = 1024.0
    (loss * loss_scale).backward()

    squared_norm = 0.0
    gradient_tensors = 0
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            assert parameter.grad is not None, f"Missing gradient: {name}"
            assert torch.isfinite(parameter.grad).all(), name
            norm = parameter.grad.float().norm().item() / loss_scale
            squared_norm += norm * norm
            gradient_tensors += 1
        else:
            assert parameter.grad is None, name

    assert squared_norm > 0, "All adapter gradients are zero."
    torch.cuda.synchronize(DEVICE)

    smoke_report = {
        "batch_size": len(check_indices),
        "padded_sequence_length": batch["input_ids"].shape[1],
        "loss": loss.item(),
        "unscaled_adapter_gradient_norm": squared_norm ** 0.5,
        "adapter_tensors_with_gradients": gradient_tensors,
        "peak_allocated_gib":
            torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
        "optimizer_updates": 0,
    }
finally:
    model.zero_grad(set_to_none=True)
    model.train(previous_mode)
    set_seed(TRAIN_SEED)
    del batch
    torch.cuda.empty_cache()

print(json.dumps(smoke_report, indent=2))
print("Forward/backward check passed; no optimizer update performed.")
print("No files written or validation/test evaluation performed.")

{
  "batch_size": 8,
  "padded_sequence_length": 339,
  "loss": 1.0753930807113647,
  "unscaled_adapter_gradient_norm": 3.012117016873611,
  "adapter_tensors_with_gradients": 448,
  "peak_allocated_gib": 6.934342861175537,
  "optimizer_updates": 0
}
Forward/backward check passed; no optimizer update performed.
No files written or validation/test evaluation performed.


In [7]:
import gc
import math
import json
import time
import hashlib
from pathlib import Path

import torch
import pandas as pd
from safetensors.torch import load_file
from peft import (
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from transformers import (
    Trainer, TrainingArguments, GenerationConfig, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

RUN_DIR = OUTPUT_DIR / "SmolLM2_LoRA_full_seed42"
RUN_DIR.mkdir(parents=True, exist_ok=True)
COMPLETION_PATH = RUN_DIR / "training_complete.json"

assert not COMPLETION_PATH.exists(), (
    "Training is already complete. Use the saved best adapter."
)
assert len(full_train) == 10590
assert len(full_validation) == 1917

EPOCHS = 3
MICRO_BATCH = 8
GRAD_ACCUM = 4
LEARNING_RATE = 1e-4
EXPECTED_STEPS = EPOCHS * math.ceil(
    math.ceil(len(full_train) / MICRO_BATCH) / GRAD_ACCUM
)
WARMUP_STEPS = math.ceil(0.03 * EXPECTED_STEPS)
assert EXPECTED_STEPS == 993

for key, expected in {
    "epochs": EPOCHS,
    "micro_batch_size": MICRO_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "seed": TRAIN_SEED,
}.items():
    assert baseline_run_config[key] == expected, (key, expected)

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)

generation_config = GenerationConfig(
    max_new_tokens=12,
    do_sample=False,
    num_beams=1,
    use_cache=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
)

initialization = json.loads(INITIALIZATION_REPORT.read_text())
adapter_path = INITIAL_ADAPTER_DIR / "adapter_model.safetensors"
assert sha256(adapter_path) == initialization["initial_adapter_sha256"]

run_config = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "method": "full_data",
    "training_rows": len(full_train),
    "validation_rows": len(full_validation),
    "training_fingerprint": full_train._fingerprint,
    "validation_fingerprint": full_validation._fingerprint,
    "input_config_sha256": sha256(OUTPUT_DIR / "input_config.json"),
    "initial_adapter_sha256": sha256(adapter_path),
    "epochs": EPOCHS,
    "micro_batch_size": MICRO_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch_size": 32,
    "learning_rate": LEARNING_RATE,
    "seed": TRAIN_SEED,
    "sequence_limit": SEQUENCE_LIMIT,
    "optimizer": "adamw_torch",
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "scheduler": "linear",
    "warmup_steps": WARMUP_STEPS,
    "expected_optimizer_steps": EXPECTED_STEPS,
    "selection_metric": "validation_macro_f1",
    "evaluation_batch_size": 8,
    "generation_config": generation_config.to_dict(),
    "initialization": initialization,
}
config_path = RUN_DIR / "run_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == run_config
else:
    save_json(config_path, run_config)

RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(run_config, sort_keys=True).encode()
).hexdigest()

label_lookup = {label.casefold(): label for label in LABELS}

def parse_prediction(response):
    normalized = " ".join(response.strip().split()).casefold()
    return label_lookup.get(normalized, "INVALID")

def evaluate_validation(eval_model, step):
    folder = RUN_DIR / "validation" / f"step_{step}"
    folder.mkdir(parents=True, exist_ok=True)
    frame = raw_frames["validation"]
    previous_padding = tokenizer.padding_side
    previous_mode = eval_model.training
    records = []
    started = time.perf_counter()

    try:
        tokenizer.padding_side = "left"
        eval_model.eval()

        with torch.inference_mode():
            for start in range(0, len(frame), 8):
                rows = frame.iloc[start:start + 8]
                prompts = [
                    tokenizer.apply_chat_template(
                        make_messages(row.claim, row.evidence),
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for row in rows.itertuples(index=False)
                ]
                inputs = tokenizer(
                    prompts,
                    padding=True,
                    truncation=False,
                    add_special_tokens=False,
                    return_tensors="pt",
                ).to(DEVICE)

                width = inputs["input_ids"].shape[1]
                assert width + 12 <= SEQUENCE_LIMIT

                generated = eval_model.generate(
                    **inputs, generation_config=generation_config
                )
                responses = tokenizer.batch_decode(
                    generated[:, width:], skip_special_tokens=True
                )

                for uid, gold, response in zip(
                    rows["id"], rows["label"], responses
                ):
                    records.append({
                        "id": uid,
                        "true_label": gold,
                        "predicted_label": parse_prediction(response),
                        "raw_response": response,
                    })
                del inputs, generated

        assert [r["id"] for r in records] == frame["id"].tolist()
        gold = [r["true_label"] for r in records]
        predicted = [r["predicted_label"] for r in records]

        metrics = {
            "accuracy": float(accuracy_score(gold, predicted)),
            "macro_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="macro", zero_division=0,
            )),
            "weighted_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="weighted", zero_division=0,
            )),
            "invalid_rate": sum(p == "INVALID" for p in predicted) / len(gold),
            "runtime": time.perf_counter() - started,
        }

        with (folder / "predictions.jsonl").open(
            "w", encoding="utf-8"
        ) as handle:
            for record in records:
                handle.write(json.dumps(record) + "\n")

        save_json(folder / "classification_report.json",
                  classification_report(
                      gold, predicted, labels=LABELS,
                      output_dict=True, zero_division=0,
                  ))
        matrix_labels = LABELS + ["INVALID"]
        pd.DataFrame(
            confusion_matrix(gold, predicted, labels=matrix_labels),
            index=matrix_labels,
            columns=matrix_labels,
        ).loc[LABELS].to_csv(folder / "confusion_matrix.csv")

        save_json(folder / "metrics.json", {
            **metrics,
            "n_examples": len(records),
            "run_config_sha256": RUN_FINGERPRINT,
        })
        return metrics
    finally:
        tokenizer.padding_side = previous_padding
        eval_model.train(previous_mode)
        gc.collect()
        torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class HealthVerTrainer(Trainer):
    def evaluate(
        self, eval_dataset=None, ignore_keys=None,
        metric_key_prefix="eval", **kwargs,
    ):
        result = evaluate_validation(self.model, self.state.global_step)
        metrics = {
            f"{metric_key_prefix}_{key}": value
            for key, value in result.items()
        }
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(
            self.args, self.state, self.control, metrics
        )
        return metrics

# Explicitly restore and verify the saved initial adapter.
# If resuming, Trainer subsequently restores the training checkpoint.
initial_state = load_file(str(adapter_path), device="cpu")
set_peft_model_state_dict(model, initial_state, adapter_name="default")
actual_state = get_peft_model_state_dict(model)
assert set(actual_state) == set(initial_state)
assert all(
    torch.equal(actual_state[key].detach().cpu(), initial_state[key])
    for key in initial_state
)
del initial_state, actual_state

model.zero_grad(set_to_none=True)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config = generation_config
model.gradient_checkpointing_disable()
tokenizer.padding_side = "right"
set_seed(TRAIN_SEED)

args = SingleGPUArguments(
    output_dir=str(RUN_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=False,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    logging_first_step=True,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
    label_names=["labels"],
)
assert args.world_size == 1
assert args.device == DEVICE
assert args.train_batch_size == MICRO_BATCH
(RUN_DIR / "training_args.json").write_text(
    args.to_json_string(), encoding="utf-8"
)

trainer = HealthVerTrainer(
    model=model,
    args=args,
    train_dataset=full_train,
    eval_dataset=full_validation,
    data_collator=training_collator,
    processing_class=tokenizer,
)
checkpoint = get_last_checkpoint(str(RUN_DIR))

print("Starting from:", checkpoint or INITIAL_ADAPTER_DIR)
print("Expected optimizer steps:", EXPECTED_STEPS)
print("Warmup steps:", WARMUP_STEPS)

torch.cuda.reset_peak_memory_stats(DEVICE)
result = trainer.train(resume_from_checkpoint=checkpoint)

assert trainer.state.global_step == EXPECTED_STEPS
assert trainer.state.epoch >= EPOCHS - 1e-6
assert trainer.state.best_metric is not None

# Trainer has restored the checkpoint selected by validation macro-F1.
BEST_ADAPTER_DIR = RUN_DIR / "best_adapter"
trainer.save_model(str(BEST_ADAPTER_DIR))
tokenizer.save_pretrained(str(BEST_ADAPTER_DIR))
trainer.save_metrics("train", result.metrics)
trainer.save_state()

completion = {
    "run_config_sha256": RUN_FINGERPRINT,
    "best_validation_macro_f1": float(trainer.state.best_metric),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "global_step": trainer.state.global_step,
    "epoch": float(trainer.state.epoch),
    "train_call_seconds": result.metrics["train_runtime"],
    "peak_allocated_gib":
        torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
    "best_adapter_sha256":
        sha256(BEST_ADAPTER_DIR / "adapter_model.safetensors"),
    "resumed_from": checkpoint,
    "runtime_note": "A resumed call excludes earlier sessions.",
}
save_json(COMPLETION_PATH, completion)

print(json.dumps(completion, indent=2))
print("Saved:", BEST_ADAPTER_DIR)
print("Full-data training complete. Test data not accessed.")

Starting from: /kaggle/working/HealthVer_SmolLM2_Experiments/initial_adapter_seed42
Expected optimizer steps: 993
Warmup steps: 30


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.213456,No log,0.567553,0.553790,0.582326,0.000000
2,0.176494,No log,0.695879,0.663506,0.700318,0.000000
3,0.140117,No log,0.721440,0.685941,0.722931,0.000000


{
  "run_config_sha256": "ecfb725811a0404f832c79ea973eb551ff4dfe35f688e51ee6d1b5b65d07bf4d",
  "best_validation_macro_f1": 0.6859411973738236,
  "best_checkpoint": "/kaggle/working/HealthVer_SmolLM2_Experiments/SmolLM2_LoRA_full_seed42/checkpoint-993",
  "global_step": 993,
  "epoch": 3.0,
  "train_call_seconds": 1968.9492,
  "peak_allocated_gib": 6.949597358703613,
  "best_adapter_sha256": "98de4d304b410e2d40c1fff4c92ad416126ecc79ae4808164fccbbb74096eb9f",
  "resumed_from": null,
  "runtime_note": "A resumed call excludes earlier sessions."
}
Saved: /kaggle/working/HealthVer_SmolLM2_Experiments/SmolLM2_LoRA_full_seed42/best_adapter
Full-data training complete. Test data not accessed.


In [8]:
import json
import time
import hashlib
import pandas as pd
import torch

from safetensors.torch import load_file
from peft import get_peft_model_state_dict
from transformers import GenerationConfig
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

TEST_DIR = RUN_DIR / "test"
TEST_METRICS_PATH = TEST_DIR / "metrics.json"

def run_full_test():
    completed = json.loads(COMPLETION_PATH.read_text())
    saved_config = json.loads((RUN_DIR / "run_config.json").read_text())
    config_hash = hashlib.sha256(
        json.dumps(saved_config, sort_keys=True).encode()
    ).hexdigest()
    assert config_hash == completed["run_config_sha256"]

    best_path = RUN_DIR / "best_adapter" / "adapter_model.safetensors"
    best_hash = sha256(best_path)
    assert best_hash == completed["best_adapter_sha256"]

    # Reuse a completed evaluation instead of evaluating again.
    if TEST_METRICS_PATH.exists():
        metrics = json.loads(TEST_METRICS_PATH.read_text())
        assert metrics["run_config_sha256"] == config_hash
        assert metrics["best_adapter_sha256"] == best_hash
        assert metrics["test_csv_sha256"] == (
            dataset_metadata["files"]["test"]["sha256"]
        )
        print("Previously completed test evaluation found.")
        return metrics

    # Verify Trainer actually restored the saved best adapter.
    test_model = trainer.model
    saved_state = load_file(str(best_path), device="cpu")
    live_state = get_peft_model_state_dict(test_model)
    assert set(saved_state) == set(live_state)
    assert all(
        torch.equal(saved_state[key], live_state[key].detach().cpu())
        for key in saved_state
    ), "Loaded adapter differs from the saved best adapter."
    del saved_state, live_state

    test_csv = (
        BASELINE_ROOT / "original_data"
        / f"{DATASET_REVISION}_healthver_test.csv"
    )
    test_hash = sha256(test_csv)
    assert test_hash == dataset_metadata["files"]["test"]["sha256"]

    test_frame = pd.read_csv(test_csv, dtype=str, keep_default_na=False)
    assert len(test_frame) == 1823
    assert test_frame["id"].is_unique
    assert set(test_frame["label"]) == set(LABELS)

    test_generation = GenerationConfig.from_dict(
        saved_config["generation_config"]
    )
    assert test_generation.do_sample is False
    assert test_generation.num_beams == 1
    assert test_generation.max_new_tokens == 12

    TEST_DIR.mkdir(parents=True, exist_ok=True)
    previous_padding = tokenizer.padding_side
    previous_mode = test_model.training
    records = []
    max_prompt_tokens = 0
    started = time.perf_counter()

    try:
        tokenizer.padding_side = "left"
        test_model.eval()

        with torch.inference_mode():
            for start in tqdm(
                range(0, len(test_frame), 8), desc="Full-data test"
            ):
                rows = test_frame.iloc[start:start + 8]
                prompts = [
                    tokenizer.apply_chat_template(
                        make_messages(row.claim, row.evidence),
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for row in rows.itertuples(index=False)
                ]
                inputs = tokenizer(
                    prompts,
                    padding=True,
                    truncation=False,
                    add_special_tokens=False,
                    return_tensors="pt",
                ).to(DEVICE)

                width = inputs["input_ids"].shape[1]
                max_prompt_tokens = max(max_prompt_tokens, width)
                assert width + 12 <= test_model.config.max_position_embeddings

                generated = test_model.generate(
                    **inputs, generation_config=test_generation
                )
                responses = tokenizer.batch_decode(
                    generated[:, width:], skip_special_tokens=True
                )

                for uid, gold, response in zip(
                    rows["id"], rows["label"], responses
                ):
                    records.append({
                        "id": uid,
                        "true_label": gold,
                        "predicted_label": parse_prediction(response),
                        "raw_response": response,
                    })
                del inputs, generated
    finally:
        tokenizer.padding_side = previous_padding
        test_model.train(previous_mode)

    assert [r["id"] for r in records] == test_frame["id"].tolist()
    gold = [r["true_label"] for r in records]
    predicted = [r["predicted_label"] for r in records]

    metrics = {
        "evaluation_split": "test",
        "n_examples": len(records),
        "run_config_sha256": config_hash,
        "best_adapter_sha256": best_hash,
        "test_csv_sha256": test_hash,
        "accuracy": float(accuracy_score(gold, predicted)),
        "macro_f1": float(f1_score(
            gold, predicted, labels=LABELS,
            average="macro", zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            gold, predicted, labels=LABELS,
            average="weighted", zero_division=0,
        )),
        "invalid_rate": sum(p == "INVALID" for p in predicted) / len(gold),
        "max_prompt_tokens": max_prompt_tokens,
        "truncation": False,
        "elapsed_seconds": time.perf_counter() - started,
    }

    with (TEST_DIR / "predictions.jsonl").open(
        "w", encoding="utf-8"
    ) as handle:
        for record in records:
            handle.write(json.dumps(record) + "\n")

    save_json(TEST_DIR / "classification_report.json",
              classification_report(
                  gold, predicted, labels=LABELS,
                  output_dict=True, zero_division=0,
              ))

    matrix_labels = LABELS + ["INVALID"]
    pd.DataFrame(
        confusion_matrix(gold, predicted, labels=matrix_labels),
        index=matrix_labels,
        columns=matrix_labels,
    ).loc[LABELS].to_csv(TEST_DIR / "confusion_matrix.csv")

    # Write the completion marker after the evaluation artifacts.
    save_json(TEST_METRICS_PATH, metrics)
    return metrics

test_metrics = run_full_test()
completed = json.loads(COMPLETION_PATH.read_text())

summary = pd.DataFrame([{
    "model": MODEL_ID,
    "method": "full_data",
    "training_rows": 10590,
    "training_seed": TRAIN_SEED,
    "best_validation_macro_f1": completed["best_validation_macro_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "test_invalid_rate": test_metrics["invalid_rate"],
    "train_call_seconds": completed["train_call_seconds"],
    "test_seconds": test_metrics["elapsed_seconds"],
}])
summary.to_csv(OUTPUT_DIR / "full_data_results.csv", index=False)

print(summary.to_string(index=False))
print("Saved:", TEST_DIR)
print("Full-data baseline complete.")

Full-data test:   0%|          | 0/228 [00:00<?, ?it/s]

                              model    method  training_rows  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
HuggingFaceTB/SmolLM2-360M-Instruct full_data          10590             42                  0.685941       0.696105       0.686229          0.695295                0.0           1968.9492     77.222901
Saved: /kaggle/working/HealthVer_SmolLM2_Experiments/SmolLM2_LoRA_full_seed42/test
Full-data baseline complete.
